# Módulo AI5 · Fine-tuning supervisado — Práctica

**Objetivo:** construir de principio a fin un SFT (*supervised fine-tuning*) para el asistente de TiendaSur, a escala de juguete para que corra en CPU, y entender cada pieza que luego hará `TRL` + `PEFT` en una GPU:

1. datos de instrucciones (calidad, división train/validación);
2. *chat template* y tokenizer (BPE entrenado aquí, sin descargas);
3. máscara de pérdida (`-100`);
4. SFT con y sin máscara, curvas de pérdida y diagnóstico de sobreajuste;
5. **LoRA implementado a mano** y comparación de parámetros entrenables;
6. esqueleto de Colab con `SFTTrainer` + QLoRA sobre un modelo abierto pequeño (no se ejecuta aquí).

Tiempo en CPU: ~2 min. No necesita clave de API ni GPU.

In [ ]:
import json, math, re, time, copy, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 7
def fijar_semillas(s=SEED):
    np.random.seed(s); torch.manual_seed(s)
fijar_semillas()
torch.set_num_threads(4)
print("torch", torch.__version__)

## 1 · Datos de instrucciones

La calidad del SFT es la calidad del dataset: **pocos ejemplos buenos, variados y sin duplicados** superan a muchos ejemplos mediocres. Dividimos *por formulación*: validación contiene preguntas reformuladas que el modelo no ha visto. Y auditamos antes de entrenar.

In [ ]:
# 12 intenciones de TiendaSur; cada una con 8 formas de preguntar y 2 formas de responder.
# "clave" = el dato que una respuesta correcta DEBE contener (lo usaremos como métrica objetiva).
INTENCIONES = {
    "envio": dict(clave="3-5 días", resp=["El envío estándar tarda 3-5 días laborables.", "Tu pedido llega en 3-5 días laborables con envío estándar."],
        preg=["¿cuánto tarda el envío?", "¿en cuántos días llega mi pedido?", "plazo de entrega estándar", "¿cuándo me llega lo que compro?", "¿qué tarda el envío normal?", "¿cuántos días tarda el envío estándar?", "¿cuándo llega mi pedido normal?", "¿qué plazo tiene el envío?"]),
    "urgente": dict(clave="14:00", resp=["Pide antes de las 14:00 y llega al día siguiente.", "Con envío urgente llega al día siguiente si pides antes de las 14:00."],
        preg=["¿y si necesito el pedido mañana?", "¿hay envío urgente?", "quiero recibirlo al día siguiente", "¿cómo es el envío exprés?", "¿hasta qué hora pido para que llegue mañana?", "¿puedo recibir el pedido mañana con envío urgente?", "¿hasta qué hora pido el envío urgente?", "necesito el envío exprés para mañana"]),
    "factura": dict(clave="mismo día", resp=["La factura se emite el mismo día de la compra y llega por correo.", "Recibirás la factura por correo el mismo día de la compra."],
        preg=["¿cuándo llega mi factura?", "¿dónde recibo la factura?", "necesito la factura de mi pedido", "¿me mandáis factura por email?", "¿cuánto tarda en llegar la factura?", "¿cuándo me mandáis la factura?", "¿llega la factura por correo?", "¿cuándo recibo la factura de mi pedido?"]),
    "cif": dict(clave="24 h", resp=["Sí: se emite en 24 h con datos fiscales completos.", "Se emite en 24 h si nos das tus datos fiscales completos."],
        preg=["¿puedo pedir factura con CIF?", "factura a nombre de mi empresa", "¿la factura con CIF tarda más?", "necesito factura con datos fiscales", "¿emitís factura para autónomos?", "¿puedo pedir la factura con CIF de mi empresa?", "factura con CIF, ¿cuánto tarda?", "¿la factura con datos fiscales tarda más?"]),
    "plazo_dev": dict(clave="30 días", resp=["Tienes 30 días naturales para devolver, con el producto sin usar.", "Puedes devolver en 30 días naturales si el producto está sin usar."],
        preg=["¿hasta cuándo puedo devolver?", "¿cuántos días tengo para devolver?", "plazo de devolución", "¿puedo devolver una compra de hace tres semanas?", "¿cuánto tiempo hay para devolver un artículo?", "¿hasta cuándo puedo devolver un pedido?", "¿cuántos días tengo para la devolución?", "plazo para devolver un producto"]),
    "coste_dev": dict(clave="cliente", resp=["La devolución la paga el cliente, salvo producto defectuoso.", "El cliente paga el envío de la devolución, salvo producto defectuoso."],
        preg=["¿quién paga la devolución?", "¿cuánto cuesta devolver?", "¿tengo que pagar el envío de vuelta?", "gastos de la devolución", "¿la devolución es gratis?", "¿quién paga los gastos de la devolución?", "¿cuánto cuesta el envío de la devolución?", "¿la devolución la pago yo?"]),
    "defectuoso": dict(clave="nosotros", resp=["Si llega defectuoso, el envío de vuelta lo pagamos nosotros.", "Un producto defectuoso se devuelve a cargo nuestro: pagamos nosotros."],
        preg=["¿quién paga si el producto llega defectuoso?", "me llegó roto, ¿qué hago?", "¿y si el producto viene defectuoso?", "producto defectuoso, ¿cómo lo devuelvo?", "llegó con un defecto, ¿lo paga la tienda?", "¿quién paga la devolución si el producto es defectuoso?", "el producto llegó defectuoso, ¿qué hago?", "¿si llega roto quién paga el envío de vuelta?"]),
    "usado": dict(clave="sin usar", resp=["No: para devolver debe estar sin usar y dentro de los 30 días.", "Solo se acepta devolución de producto sin usar."],
        preg=["¿puedo devolver un producto usado?", "ya abrí y probé el producto, ¿lo devuelvo?", "devolución de algo usado", "¿aceptáis devoluciones de artículos usados?", "lo estrené, ¿puedo devolverlo?", "¿puedo devolver un producto que ya usé?", "¿se puede devolver algo usado?", "lo probé una vez, ¿puedo devolverlo?"]),
    "pago": dict(clave="tarjeta", resp=["Aceptamos tarjeta, transferencia y Bizum.", "Puedes pagar con tarjeta, transferencia o Bizum."],
        preg=["¿qué formas de pago aceptáis?", "¿puedo pagar con Bizum?", "¿se paga con tarjeta?", "métodos de pago", "¿aceptáis transferencia?", "¿qué formas de pago tenéis?", "¿se puede pagar con tarjeta o Bizum?", "¿aceptáis tarjeta de crédito?"]),
    "seguimiento": dict(clave="enlace", resp=["Recibirás un enlace de seguimiento por correo al enviarse el pedido.", "Te enviamos por correo un enlace para seguir tu pedido."],
        preg=["¿cómo sigo mi pedido?", "¿dónde veo el seguimiento?", "quiero rastrear mi envío", "¿tenéis número de seguimiento?", "¿cómo sé dónde está mi paquete?", "¿cómo hago el seguimiento de mi pedido?", "¿me dais un enlace de seguimiento?", "quiero seguir mi paquete"]),
    "garantia": dict(clave="2 años", resp=["Todos los productos tienen 2 años de garantía legal.", "La garantía legal es de 2 años en todos los productos."],
        preg=["¿qué garantía tienen los productos?", "¿cuántos años de garantía hay?", "si se estropea, ¿tiene garantía?", "garantía de la cafetera", "¿cubre la garantía las averías?", "¿qué garantía tiene un producto?", "¿cuántos años de garantía tiene la cafetera?", "¿hay garantía si se estropea?"]),
    "cambio": dict(clave="cambio", resp=["Sí, puedes pedir un cambio por otro producto dentro de 30 días.", "Se puede solicitar un cambio por otro artículo en 30 días."],
        preg=["¿puedo cambiar un producto por otro?", "quiero un cambio de modelo", "¿hacéis cambios?", "me equivoqué de taza, ¿la cambio?", "¿se puede cambiar lo que compré?", "¿puedo hacer un cambio de producto?", "quiero cambiar un producto por otro", "¿aceptáis cambios de modelo?"]),
}
SISTEMA = "Eres el asistente de TiendaSur. Responde breve y con datos exactos."

def construir_pares():
    """Devuelve (train, val): las 6 primeras preguntas de cada intención a train, las 2 últimas a val
    (val = formulaciones que el modelo NO ha visto, de intenciones que SÍ conoce)."""
    train, val = [], []
    for nombre, d in INTENCIONES.items():
        for i, q in enumerate(d["preg"]):
            par = dict(intencion=nombre, pregunta=q, respuesta=d["resp"][i % 2], clave=d["clave"])
            (train if i < 6 else val).append(par)
    return train, val

TRAIN, VAL = construir_pares()
print(len(TRAIN), "pares de entrenamiento;", len(VAL), "de validación")
pd.DataFrame(TRAIN[:4])[["pregunta", "respuesta", "clave"]]

In [ ]:
def normalizar(t: str) -> str:
    return re.sub(r"\W+", " ", t.lower()).strip()

def auditar(train, val):
    """Comprobaciones mínimas ANTES de entrenar. Un dataset sucio da un modelo sucio."""
    inf = {}
    inf["vacios"] = sum(1 for p in train + val if not p["pregunta"].strip() or not p["respuesta"].strip())
    inf["duplicados_en_train"] = len(train) - len({normalizar(p["pregunta"]) for p in train})
    inf["fuga_train_val"] = len({normalizar(p["pregunta"]) for p in train} & {normalizar(p["pregunta"]) for p in val})
    longs = [len(p["respuesta"]) for p in train]
    inf["resp_chars_min_med_max"] = (min(longs), int(np.median(longs)), max(longs))
    inf["intenciones_en_train"] = len({p["intencion"] for p in train})
    return inf

print(auditar(TRAIN, VAL))

In [ ]:
SALUDOS = ["", "hola, ", "buenas, ", "una duda: ", "por favor, ", "oye, "]
CIERRES = ["", " gracias", " porfa", ""]

def aumentar(pares, n, semilla=0):
    """Aumento de datos por variaciones de superficie (saludo, cortesía). NO inventa información nueva:
    la respuesta es la misma. Sirve para que el modelo no se ate a la redacción exacta de la pregunta."""
    g = np.random.default_rng(semilla); salida = []
    for i in range(n):
        p = dict(pares[i % len(pares)]); q = p["pregunta"]
        abre, cierra = ("¿", "?") if q.startswith("¿") else ("", "")
        p["pregunta"] = SALUDOS[g.integers(len(SALUDOS))] + abre + q.strip("¿?") + cierra + CIERRES[g.integers(len(CIERRES))]
        salida.append(p)
    return salida

TRAIN_AUG = aumentar(TRAIN, 6 * len(TRAIN))      # 432 ejemplos; la validación NO se aumenta
print(len(TRAIN_AUG), "ejemplos tras el aumento. Ejemplos:")
for p in TRAIN_AUG[:3]: print("  ", p["pregunta"], "->", p["respuesta"][:40])

## 2 · Chat template y tokenizer

Un modelo de lenguaje solo ve una secuencia de números. El **chat template** es la receta que convierte una conversación (`system` / `user` / `assistant`) en el texto exacto —con *tokens especiales* que marcan dónde empieza y acaba cada turno— que el modelo vio durante el entrenamiento. Los modelos abiertos publican su plantilla en `tokenizer.chat_template` (una plantilla Jinja) y la aplicas con `tokenizer.apply_chat_template(...)`.

Aquí hacemos lo mismo en pequeño: una plantilla estilo ChatML con **Jinja** y un tokenizer **BPE real** (librería `tokenizers`, la misma de Hugging Face) entrenado sobre nuestro corpus. Así no descargamos nada y el mecanismo es idéntico. Si `tokenizers` no estuviera instalado, caemos a un tokenizer por caracteres.

In [ ]:
from jinja2 import Template

PLANTILLA = (
    "{% for m in messages %}<|im_start|>{{ m['role'] }}\n{{ m['content'] }}<|im_end|>\n{% endfor %}"
    "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"
)
ESPECIALES = ["<|pad|>", "<|im_start|>", "<|im_end|>"]

def aplicar_plantilla(messages, add_generation_prompt=False, plantilla=PLANTILLA):
    return Template(plantilla).render(messages=messages, add_generation_prompt=add_generation_prompt)

def mensajes(par, con_respuesta=True):
    m = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": par["pregunta"]}]
    if con_respuesta:
        m.append({"role": "assistant", "content": par["respuesta"]})
    return m

print(repr(aplicar_plantilla(mensajes(TRAIN[0]))))

In [ ]:
corpus = [aplicar_plantilla(mensajes(p)) for p in TRAIN_AUG + VAL]

class TokBPE:
    """Tokenizer BPE entrenado en local (ByteLevel => decodifica sin perder espacios ni tildes)."""
    def __init__(self, textos, vocab=400):
        from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
        self.t = Tokenizer(models.BPE())
        self.t.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        self.t.decoder = decoders.ByteLevel()
        tr = trainers.BpeTrainer(vocab_size=vocab, special_tokens=ESPECIALES,
                                 initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
        self.t.train_from_iterator(textos, tr)
        self.nombre = "BPE (tokenizers)"
    def encode(self, s): return self.t.encode(s).ids
    def decode(self, ids): return self.t.decode(ids, skip_special_tokens=False)
    def tokens(self, s): return self.t.encode(s).tokens
    def id(self, tok): return self.t.token_to_id(tok)
    @property
    def n(self): return self.t.get_vocab_size()

class TokChar:
    """Plan B: un id por carácter + los especiales (misma interfaz)."""
    def __init__(self, textos, vocab=None):
        plano = sorted(set("".join(textos).replace("<|im_start|>", "").replace("<|im_end|>", "").replace("<|pad|>", "")))
        self.vocab = ESPECIALES + plano; self.idx = {t: i for i, t in enumerate(self.vocab)}
        self.nombre = "caracteres"
    def _parte(self, s): return [x for x in re.split(r"(<\|pad\|>|<\|im_start\|>|<\|im_end\|>)", s) if x]
    def tokens(self, s): return [y for x in self._parte(s) for y in ([x] if x in ESPECIALES else list(x))]
    def encode(self, s): return [self.idx[t] for t in self.tokens(s)]
    def decode(self, ids): return "".join(self.vocab[i] for i in ids)
    def id(self, tok): return self.idx[tok]
    @property
    def n(self): return len(self.vocab)

try:
    TOK = TokBPE(corpus)
except ImportError:
    TOK = TokChar(corpus)
PAD, INI, FIN = TOK.id("<|pad|>"), TOK.id("<|im_start|>"), TOK.id("<|im_end|>")
V = TOK.n
ej = aplicar_plantilla(mensajes(TRAIN[0]))
print(f"tokenizer: {TOK.nombre} | vocabulario: {V} | tokens del ejemplo: {len(TOK.encode(ej))}")
print(TOK.tokens(ej)[:24], "…")
assert TOK.decode(TOK.encode(ej)) == ej, "el tokenizer debe ser reversible"

## 3 · Máscara de pérdida

En SFT el modelo predice **cada token** a partir de los anteriores, pero solo queremos que *aprenda a responder*. Por eso las posiciones del prompt (sistema + pregunta + cabecera `assistant`) llevan la etiqueta `-100`, que `F.cross_entropy(..., ignore_index=-100)` ignora.

Construimos `ids` (entrada) y `labels` (objetivo) con la regla estándar: `input = ids[:-1]`, `target = ids[1:]`, y enmascaramos el target mientras estemos dentro del prompt. Aprendemos la respuesta **y** el token `<|im_end|>` (así el modelo aprende a parar).

In [ ]:
def codificar(par, mascara=True):
    """(pregunta, respuesta) -> (ids, labels) ya desplazados. labels = -100 donde no se aprende."""
    prompt = aplicar_plantilla(mensajes(par, con_respuesta=False), add_generation_prompt=True)
    ids_p = TOK.encode(prompt)
    ids_r = TOK.encode(par["respuesta"] + "<|im_end|>")
    ids = ids_p + ids_r
    objetivo = ids[1:]
    if mascara:
        labels = [-100] * (len(ids_p) - 1) + ids_r        # el primer token de la respuesta es el target nº len(ids_p)-1
    else:
        labels = list(objetivo)                            # aprende TODO el texto
    return ids[:-1], labels, len(ids_p)

def verificar_mascara(par):
    ids, labels, n_prompt = codificar(par)
    entrenados = [l for l in labels if l != -100]
    texto = TOK.decode(entrenados)
    assert texto == par["respuesta"] + "<|im_end|>", f"la máscara no coincide con la respuesta: {texto!r}"
    return len(ids), len(entrenados)

total, aprendidos = verificar_mascara(TRAIN[0])
print(f"secuencia de {total} tokens; se aprende de {aprendidos} ({aprendidos/total:.0%}); el resto es prompt enmascarado")

ids, labels, _ = codificar(TRAIN[0])
tabla = pd.DataFrame({"token de entrada": [TOK.decode([i]) for i in ids], "objetivo (label)": [TOK.decode([l]) if l != -100 else "·  (-100)" for l in labels]})
tabla.iloc[len(ids)-aprendidos-3: len(ids)-aprendidos+6]

In [ ]:
def lote(pares, mascara=True, ctx=128):
    """Agrupa pares en un lote con relleno (padding). Las posiciones de relleno también valen -100."""
    cod = [codificar(p, mascara) for p in pares]
    T = min(max(len(c[0]) for c in cod), ctx)
    x = torch.full((len(cod), T), PAD, dtype=torch.long)
    y = torch.full((len(cod), T), -100, dtype=torch.long)
    for i, (ids, lab, _) in enumerate(cod):
        ids, lab = ids[:T], lab[:T]
        x[i, :len(ids)] = torch.tensor(ids); y[i, :len(lab)] = torch.tensor(lab)
    return x, y

x, y = lote(TRAIN[:4])
print("lote:", tuple(x.shape), "| tokens con pérdida:", int((y != -100).sum()), "de", y.numel())

## 4 · El modelo: un transformer decoder pequeño

Es la arquitectura del módulo AI1 (atención causal + MLP, *pre-LayerNorm*) con ~100 k parámetros. Hacemos dos pasos, igual que en la vida real:

1. **«Preentrenamiento»**: el modelo lee texto (aquí, los textos de entrenamiento y un poco de texto general) y aprende lengua y formato, sin máscara.
2. **SFT**: partimos de ese modelo «base» y lo ajustamos con pares (instrucción, respuesta).

Medimos todo con la misma vara: la **pérdida sobre los tokens de respuesta del conjunto de validación** (preguntas reformuladas que no vio).

In [ ]:
class Bloque(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.h = heads
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.qkv = nn.Linear(dim, 3 * dim); self.proj = nn.Linear(dim, dim)
        self.fc1 = nn.Linear(dim, 4 * dim); self.fc2 = nn.Linear(4 * dim, dim)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, -1).transpose(1, 2) for t in (q, k, v))
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(a.transpose(1, 2).reshape(B, T, C))
        return x + self.fc2(F.gelu(self.fc1(self.ln2(x))))

class MiniLM(nn.Module):
    def __init__(self, vocab, dim=64, capas=2, heads=4, ctx=128):
        super().__init__()
        self.ctx = ctx
        self.tok, self.pos = nn.Embedding(vocab, dim), nn.Embedding(ctx, dim)
        self.bloques = nn.ModuleList([Bloque(dim, heads) for _ in range(capas)])
        self.ln = nn.LayerNorm(dim); self.head = nn.Linear(dim, vocab, bias=False)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1]))
        for b in self.bloques: x = b(x)
        return self.head(self.ln(x))

def n_params(m, solo_entrenables=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not solo_entrenables))

@torch.no_grad()
def perdida_respuesta(m, pares):
    """Cross-entropy media SOLO sobre tokens de respuesta (misma vara para todos los modelos)."""
    m.eval(); x, y = lote(pares, mascara=True)
    return F.cross_entropy(m(x).reshape(-1, V), y.reshape(-1), ignore_index=-100).item()

def entrenar(m, train, val, pasos, lr, mascara=True, bs=8, eval_cada=20, params=None, semilla=SEED):
    """Bucle SFT. Devuelve las curvas (paso, loss_train, loss_val) para diagnosticar."""
    g = np.random.default_rng(semilla)
    opt = torch.optim.AdamW(params if params is not None else m.parameters(), lr=lr, weight_decay=0.0)
    curva = []
    for paso in range(1, pasos + 1):
        m.train()
        x, y = lote([train[i] for i in g.integers(0, len(train), bs)], mascara)
        loss = F.cross_entropy(m(x).reshape(-1, V), y.reshape(-1), ignore_index=-100)
        opt.zero_grad(); loss.backward(); opt.step()
        if paso % eval_cada == 0 or paso == 1:
            curva.append((paso, perdida_respuesta(m, train[::max(1, len(train) // 72)]), perdida_respuesta(m, val)))
    return pd.DataFrame(curva, columns=["paso", "train", "val"])

# --- 1) «preentrenamiento»: texto plano sobre train + frases generales, sin máscara
GENERAL = ["El café de especialidad se muele justo antes de prepararlo.", "Una buena cafetera de émbolo necesita agua a 93 grados.",
           "La cerámica artesanal se limpia con agua tibia y jabón neutro.", "Las tazas de gres aguantan el lavavajillas.",
           "El té verde se infusiona menos tiempo que el negro.", "Guarda el café en un recipiente hermético y oscuro."]
t0 = time.time(); fijar_semillas()
BASE = MiniLM(V)
# El base conoce la lengua, el formato y los DATOS de la tienda (las respuestas), pero no sabe qué responder a cada pregunta:
# eso es justo lo que enseña el SFT. Por eso la pregunta de preentrenamiento es siempre la misma.
RESPUESTAS = sorted({r for d in INTENCIONES.values() for r in d["resp"]})
pares_pre = [dict(pregunta="Cuéntame algo.", respuesta=t) for t in RESPUESTAS + GENERAL]
cur_pre = entrenar(BASE, pares_pre, VAL, pasos=300, lr=3e-3, mascara=False, bs=12, eval_cada=100)
print(f"modelo base: {n_params(BASE):,} parámetros | preentrenado en {time.time()-t0:.0f} s")
print(cur_pre.round(3).to_string(index=False))

## 5 · SFT con máscara frente a SFT sin máscara

Mismo modelo base, mismos datos, mismos pasos. La única diferencia: contra qué tokens se calcula la pérdida. Después comparamos con la vara común (pérdida de respuesta en validación) y con una métrica objetiva: **¿la respuesta generada contiene el dato clave?** (por ejemplo «14:00» o «30 días»).

In [ ]:
@torch.no_grad()
def generar(m, pregunta, max_nuevos=60):
    """Decodificación voraz (greedy): determinista, ideal para evaluar."""
    m.eval()
    ids = TOK.encode(aplicar_plantilla(mensajes(dict(pregunta=pregunta), con_respuesta=False), add_generation_prompt=True))
    for _ in range(max_nuevos):
        sig = m(torch.tensor([ids[-m.ctx:]]))[0, -1].argmax().item()
        if sig == FIN: break
        ids.append(sig)
    return TOK.decode(ids[len(TOK.encode(aplicar_plantilla(mensajes(dict(pregunta=pregunta), con_respuesta=False), add_generation_prompt=True))):])

def tasa_dato_clave(m, pares):
    return float(np.mean([p["clave"].lower() in generar(m, p["pregunta"]).lower() for p in pares]))

def sft(mascara, pasos=600, lr=3e-3, bs=16):
    fijar_semillas(); m = copy.deepcopy(BASE)
    curva = entrenar(m, TRAIN_AUG, VAL, pasos, lr, mascara=mascara, bs=bs)
    return m, curva

m_mask, cur_mask = sft(True)
m_nomask, cur_nomask = sft(False)
res = pd.DataFrame({
    "val loss (respuesta)": [perdida_respuesta(BASE, VAL), perdida_respuesta(m_mask, VAL), perdida_respuesta(m_nomask, VAL)],
    "dato clave en val": [tasa_dato_clave(BASE, VAL), tasa_dato_clave(m_mask, VAL), tasa_dato_clave(m_nomask, VAL)],
}, index=["base (sin SFT)", "SFT con máscara", "SFT sin máscara"]).round(3)
print(res)
print(f"\nTokens con pérdida por ejemplo: con máscara {np.mean([sum(l != -100 for l in codificar(p, True)[1]) for p in TRAIN]):.0f}, sin máscara {np.mean([sum(l != -100 for l in codificar(p, False)[1]) for p in TRAIN]):.0f}")
print('Lectura: aquí la diferencia en val es pequeña (prompts cortos). La máscara importa cuando el prompt es largo (RAG con miles de tokens de contexto) o ruidoso: sin ella, la mayor parte del gradiente se gasta en aprender a predecir el prompt.')
for q in [VAL[0]["pregunta"], VAL[6]["pregunta"]]:
    print("\nP:", q)
    print("  base         ->", repr(generar(BASE, q)[:90]))
    print("  con máscara  ->", repr(generar(m_mask, q)[:90]))
    print("  sin máscara  ->", repr(generar(m_nomask, q)[:90]))

## 6 · Curvas de pérdida y diagnóstico de sobreajuste

Mirar solo la pérdida de entrenamiento engaña: siempre baja. Lo que importa es la **pérdida de validación**. Patrones típicos:

| Curvas | Diagnóstico | Remedio |
|---|---|---|
| train y val bajan juntas | aprendiendo bien | seguir |
| train baja, val **sube** tras un mínimo | sobreajuste (memoriza) | parar en el mínimo (*early stopping*), menos épocas/lr, más datos, más variedad, regularización |
| ambas altas y planas | infraajuste | más pasos, más lr, más capacidad (r mayor) |
| val oscila mucho | lr alto o lotes pequeños | bajar lr, aumentar lote |

Forzamos el sobreajuste con pocos ejemplos y muchos pasos, y escribimos una función que lo diagnostica sola.

In [ ]:
def diagnostico(curva, tol=0.05):
    """Lee las curvas y dice qué pasa. Reglas simples y explícitas, para poder discutirlas."""
    i_min = curva["val"].idxmin(); paso_min = int(curva.loc[i_min, "paso"])
    val_min, val_fin = float(curva.loc[i_min, "val"]), float(curva["val"].iloc[-1])
    brecha = val_fin - float(curva["train"].iloc[-1])
    if val_fin > val_min * (1 + tol) and paso_min < curva["paso"].iloc[-1]:
        v = f"SOBREAJUSTE: la validación toca fondo en el paso {paso_min} ({val_min:.3f}) y acaba en {val_fin:.3f}. Para en el paso {paso_min}."
    elif curva["val"].iloc[-1] > 0.8 * curva["val"].iloc[0]:
        v = "INFRAAJUSTE: la validación apenas baja. Más pasos, más lr o más capacidad."
    else:
        v = "OK: la validación sigue bajando o está estable."
    return dict(paso_min_val=paso_min, val_min=round(val_min, 3), val_final=round(val_fin, 3), brecha_final=round(brecha, 3), veredicto=v)

# Pocos datos (12 pares sin aumentar) y muchos pasos => memoriza
fijar_semillas(); m_pocos = copy.deepcopy(BASE)
cur_pocos = entrenar(m_pocos, TRAIN[::6], VAL, pasos=600, lr=2e-3, eval_cada=30)
# Todos los datos (72 pares + aumento = 432 ejemplos), mismos pasos
fijar_semillas(); m_todos = copy.deepcopy(BASE)
cur_todos = entrenar(m_todos, TRAIN_AUG, VAL, pasos=600, lr=2e-3, eval_cada=30)

fig, ejes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, cur, tit in zip(ejes, (cur_pocos, cur_todos), ("12 pares de train", "72 pares + aumento")):
    ax.plot(cur.paso, cur.train, label="train"); ax.plot(cur.paso, cur.val, label="validación")
    ax.set_title(tit); ax.set_xlabel("paso"); ax.legend()
ejes[0].set_ylabel("pérdida (respuesta)"); plt.tight_layout(); plt.show()
for nombre, cur in (("12 pares", cur_pocos), ("72 pares + aumento", cur_todos)):
    print(nombre, "->", diagnostico(cur))

## 7 · LoRA implementado a mano

**Idea:** en lugar de actualizar una matriz grande `W` (`out × in`), la congelamos y aprendemos una corrección de bajo rango:

$$W' = W + \frac{\alpha}{r}\, B A, \qquad A \in \mathbb{R}^{r \times in},\; B \in \mathbb{R}^{out \times r},\; r \ll \min(in, out)$$

- Parámetros entrenables: `r·(in+out)` en vez de `out·in`.
- `B` empieza en **cero**: al principio `W' = W`, el modelo no cambia. `A` empieza aleatorio pequeño (si ambas fueran cero, el gradiente sería cero).
- `α/r` escala la corrección; subir `r` sin tocar `α` reduce el efecto por parámetro (por eso se suele usar `α = 2r` o `α = r`).
- Tras entrenar, se puede **fusionar** (`merge`): `W ← W + (α/r)BA`, sin coste extra en inferencia.

Lo implementamos como un módulo que envuelve un `nn.Linear`.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=4, alpha=8):
        super().__init__()
        self.base, self.r, self.escala = base, r, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.02)   # (r, in)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))          # (out, r): arranca en 0
        for p in self.base.parameters(): p.requires_grad = False
    def forward(self, x):
        return self.base(x) + self.escala * (x @ self.A.T) @ self.B.T
    @torch.no_grad()
    def pesos_fusionados(self):
        return self.base.weight + self.escala * (self.B @ self.A)

def aplicar_lora(modelo, r=4, alpha=8, objetivos=("qkv", "proj")):
    """Congela TODO el modelo y sustituye los Linear indicados por LoRALinear."""
    for p in modelo.parameters(): p.requires_grad = False
    for b in modelo.bloques:
        for nombre in objetivos:
            setattr(b, nombre, LoRALinear(getattr(b, nombre), r, alpha))
    return modelo

# --- Verificación 1: al inicio LoRA no cambia el modelo (B = 0)
fijar_semillas(); m_l = aplicar_lora(copy.deepcopy(BASE), r=4, alpha=8)
x_t, _ = lote(VAL[:3])
assert torch.allclose(m_l(x_t), BASE(x_t), atol=1e-6), "con B=0 la salida debe ser idéntica"
print("OK: con B=0 el modelo con LoRA es idéntico al base")

# --- Verificación 2: la fusión da exactamente la misma salida que el adaptador sin fusionar
with torch.no_grad():
    for b in m_l.bloques:
        b.qkv.B.normal_(0, 0.1); b.proj.B.normal_(0, 0.1)          # simulamos un adaptador ya entrenado
    y_sin_fusionar = m_l(x_t)
    capa = m_l.bloques[0].qkv; entrada = torch.randn(2, 5, 64)
    y_fus = entrada @ capa.pesos_fusionados().T + capa.base.bias
    assert torch.allclose(capa(entrada), y_fus, atol=1e-5)
print("OK: x·(W + α/r·BA)ᵀ + b == salida del adaptador sin fusionar")

In [ ]:
# Comparación de parámetros entrenables: full fine-tuning vs LoRA con distintos rangos
filas = [dict(config="full fine-tuning", entrenables=n_params(BASE), pct=100.0)]
for r in (1, 4, 16):
    ml = aplicar_lora(copy.deepcopy(BASE), r=r, alpha=2 * r)
    filas.append(dict(config=f"LoRA r={r} (qkv+proj)", entrenables=n_params(ml, True), pct=100 * n_params(ml, True) / n_params(ml)))
ml = aplicar_lora(copy.deepcopy(BASE), r=4, alpha=8, objetivos=("qkv", "proj", "fc1", "fc2"))
filas.append(dict(config="LoRA r=4 (todas las lineales)", entrenables=n_params(ml, True), pct=100 * n_params(ml, True) / n_params(ml)))
print(pd.DataFrame(filas).round(2).to_string(index=False))

# Y en un modelo de verdad (dimensiones tipo 7B: hidden 4096, 32 capas, q,k,v,o de 4096x4096)
oculto, capas, r = 4096, 32, 8
full = capas * 4 * oculto * oculto
lora = capas * 4 * r * (oculto + oculto)
print(f"\n7B (solo atención): full = {full/1e9:.2f} B parámetros | LoRA r={r} = {lora/1e6:.1f} M ({100*lora/full:.2f} %)")

### Entrenar con LoRA frente a full fine-tuning

Mismo modelo base, mismos datos y pasos. Con LoRA solo se optimizan `A` y `B` (en todas las capas lineales). Como hay muchos menos parámetros, el *learning rate* de LoRA suele ser **mayor** (aquí 1·10⁻² frente a 3·10⁻³). En esta escala diminuta el SFT completo ganará o empatará; la ventaja real de LoRA aparece con modelos grandes: memoria, tiempo de guardado (adaptador de MB, no de GB) y poder tener un adaptador por cliente sobre el mismo modelo base.

In [ ]:
def sft_lora(r=8, alpha=None, objetivos=("qkv", "proj", "fc1", "fc2"), pasos=600, lr=1e-2, bs=16):
    fijar_semillas(); m = aplicar_lora(copy.deepcopy(BASE), r=r, alpha=alpha or 2 * r, objetivos=objetivos)
    cur = entrenar(m, TRAIN_AUG, VAL, pasos, lr, bs=bs, params=[p for p in m.parameters() if p.requires_grad])
    return m, cur

t0 = time.time(); m_lora, cur_lora = sft_lora(); t_lora = time.time() - t0
t0 = time.time(); m_full, cur_full = sft(True); t_full = time.time() - t0
comp = pd.DataFrame({
    "parámetros entrenables": [n_params(m_full), n_params(m_lora, True)],
    "val loss": [perdida_respuesta(m_full, VAL), perdida_respuesta(m_lora, VAL)],
    "dato clave en val": [tasa_dato_clave(m_full, VAL), tasa_dato_clave(m_lora, VAL)],
    "segundos": [t_full, t_lora],
}, index=["SFT completo", "LoRA r=8 (todas las lineales)"]).round(3)
print(comp)

# Los pesos congelados no han cambiado ni un bit: solo se movió el adaptador
base_w = BASE.bloques[0].qkv.weight
assert torch.equal(m_lora.bloques[0].qkv.base.weight, base_w)
print("OK: W congelada intacta; el cambio vive solo en A y B")

### QLoRA: por qué cabe un 7B en una T4

QLoRA guarda la base **congelada en 4 bits** (NF4) y entrena el adaptador LoRA en precisión normal; los gradientes atraviesan la base cuantizada. Hagamos las cuentas:

In [ ]:
# Presupuesto de memoria (estimación gruesa) para ajustar un modelo de 7 000 M de parámetros
def gb(n_bytes): return n_bytes / 1e9
P = 7e9
filas = [
    ("full FT, bf16 + AdamW", gb(P * 2 + P * 2 + P * 8), "pesos 2 B + gradientes 2 B + estados Adam (fp32: 2x4 B) por parámetro"),
    ("LoRA r=8, base en bf16", gb(P * 2) + 1.0, "pesos congelados 14 GB + adaptador y sus estados (~1 GB)"),
    ("QLoRA r=8, base en 4 bits", gb(P * 0.5) + 1.0, "pesos congelados a 4 bits ~3,5 GB + adaptador (~1 GB)"),
]
print(pd.DataFrame(filas, columns=["método", "GB (sin activaciones)", "de dónde sale"]).round(1).to_string(index=False))
print("\nA esto se suman las activaciones (dependen de lote y longitud). Una T4 tiene 16 GB: solo QLoRA cabe cómodo con 7B.")

## 8 · [Colab] Esqueleto con TRL `SFTTrainer` + PEFT + QLoRA

Esta celda **no se ejecuta aquí** (necesita GPU, `bitsandbytes` e internet). Cópiala a un cuaderno de Google Colab con GPU T4. Va comentada para que el resto del notebook siga funcionando.

Antes de lanzarla, **comprueba el modelo vigente**: las familias abiertas pequeñas (Qwen, SmolLM, Gemma, Llama…) se renuevan cada pocos meses. Busca en el Hub de Hugging Face un modelo `-Instruct` de 0,5 a 3 B parámetros con licencia que te permita usarlo, y cambia `BASE_ID`. También las firmas de TRL cambian entre versiones: si algún argumento falla, mira la documentación de la versión que tengas instalada.

Fíjate en cómo se corresponde con lo que hemos hecho a mano:

| Aquí a mano | En TRL / PEFT |
|---|---|
| `aplicar_plantilla` | `tokenizer.apply_chat_template` (lo hace el trainer) |
| máscara `-100` en el prompt | formato *prompt/completion* + `completion_only_loss` (o `assistant_only_loss`) |
| `LoRALinear(r, alpha)` | `LoraConfig(r, lora_alpha, target_modules)` |
| congelar y cuantizar la base | `BitsAndBytesConfig(load_in_4bit=True)` + `prepare_model_for_kbit_training` |
| `entrenar(...)` con curvas | `SFTTrainer.train()` + `eval_dataset` + `eval_strategy="steps"` |
| guardar el adaptador | `trainer.save_model()` (solo MB) |

In [ ]:
ESQUELETO_COLAB = """
# %pip install -q -U transformers trl peft accelerate datasets bitsandbytes
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

BASE_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # COMPRUEBA el modelo vigente en el Hub (Qwen/SmolLM/Gemma/Llama, 0,5-3 B)

tok = AutoTokenizer.from_pretrained(BASE_ID)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)   # en T4 usa torch.float16
modelo = AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=bnb, device_map="auto")

# Datos en formato conversacional prompt/completion (ver el ejercicio 5): TRL aplica la plantilla del
# modelo y calcula la pérdida solo sobre la respuesta.
train = Dataset.from_list(TRAIN_TRL)      # lista de {"prompt": [...], "completion": [...]}
val = Dataset.from_list(VAL_TRL)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

args = SFTConfig(output_dir="sft-tiendasur", num_train_epochs=3, per_device_train_batch_size=4,
                 gradient_accumulation_steps=4, learning_rate=2e-4, lr_scheduler_type="cosine",
                 warmup_ratio=0.05, logging_steps=5, eval_strategy="steps", eval_steps=20,
                 max_length=512, report_to="none", seed=7)    # en versiones antiguas: max_seq_length
trainer = SFTTrainer(model=modelo, args=args, train_dataset=train, eval_dataset=val,
                     processing_class=tok, peft_config=lora)
trainer.train()
trainer.save_model("sft-tiendasur")           # guarda solo el adaptador

# Evaluación: mismas preguntas de validación, modelo base frente a modelo con adaptador (como en el AI4)
"""
print(ESQUELETO_COLAB.count("\n"), "líneas de esqueleto (no ejecutado aquí)")

---

## Ejercicios (5)

1. **Auditoría de datos.** Copia `TRAIN`, añade a mano tres defectos (un duplicado con otra capitalización, un par con respuesta vacía y una pregunta de `VAL` repetida en `TRAIN`) y escribe `auditar_v2` que los detecte. ¿Por qué la fuga train/val es el defecto más peligroso?
2. **Sobreajuste.** Entrena con `TRAIN[::6]` (12 pares) 600 pasos y aplica `diagnostico`. Después reentrena **parando en el paso mínimo** de validación y compara la pérdida de validación final.
3. **Plantilla rota.** Define una plantilla *sin* la cabecera `<|im_start|>assistant` en el prompt de generación y comprueba qué ocurre con la máscara (¿qué tokens se entrenan?). Escribe un test que falle si la máscara no empieza justo tras la cabecera del asistente.
4. **Rango y alfa.** Barrido LoRA con `r ∈ {1, 4, 16}` (con `alpha = 2r`): tabla de parámetros entrenables, pérdida de validación y dato clave. ¿A partir de qué `r` deja de compensar subirlo?
5. **Preparar Colab.** Convierte `TRAIN`/`VAL` al formato *prompt/completion* conversacional de TRL, valida su estructura con asserts y guárdalo como JSONL en memoria (no en disco). Escribe tres líneas: ¿qué mejora un SFT real frente a este juguete?

In [ ]:
# Ejercicio 1 · tu código aquí


In [ ]:
# Ejercicio 2 · tu código aquí


In [ ]:
# Ejercicio 3 · tu código aquí


In [ ]:
# Ejercicio 4 · tu código aquí


In [ ]:
# Ejercicio 5 · tu código aquí
